# L4: Tools for a Customer Outreach Campaign

In this lesson, you will learn more about Tools. You'll focus on three key elements of Tools:
- Versatility
- Fault Tolerance
- Caching

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM
- [Serper](https://serper.dev)

In [2]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
import os
from utils import get_openai_api_key, pretty_print_result
from utils import get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## Creating Agents

In [4]:
sales_rep_agent = Agent(
    role="Sales Representative",
    goal="Identify high-value leads that match "
         "our ideal customer profile",
    backstory=(
        "As a part of the dynamic sales team at CrewAI, "
        "your mission is to scour "
        "the digital landscape for potential leads. "
        "Armed with cutting-edge tools "
        "and a strategic mindset, you analyze data, "
        "trends, and interactions to "
        "unearth opportunities that others might overlook. "
        "Your work is crucial in paving the way "
        "for meaningful engagements and driving the company's growth."
    ),
    allow_delegation=False,
    verbose=True
)

In [5]:
lead_sales_rep_agent = Agent(
    role="Lead Sales Representative",
    goal="Nurture leads with personalized, compelling communications",
    backstory=(
        "Within the vibrant ecosystem of CrewAI's sales department, "
        "you stand out as the bridge between potential clients "
        "and the solutions they need."
        "By creating engaging, personalized messages, "
        "you not only inform leads about our offerings "
        "but also make them feel seen and heard."
        "Your role is pivotal in converting interest "
        "into action, guiding leads through the journey "
        "from curiosity to commitment."
    ),
    allow_delegation=False,
    verbose=True
)

## Creating Tools

### crewAI Tools

In [6]:
from crewai_tools import DirectoryReadTool, \
                         FileReadTool, \
                         SerperDevTool

In [7]:
directory_read_tool = DirectoryReadTool(directory='./instructions')
file_read_tool = FileReadTool()
search_tool = SerperDevTool()

### Custom Tool
- Create a custom tool using crewAi's [BaseTool](https://docs.crewai.com/core-concepts/Tools/#subclassing-basetool) class

In [8]:
from crewai_tools import BaseTool

- Every Tool needs to have a `name` and a `description`.
- For simplicity and classroom purposes, `SentimentAnalysisTool` will return `positive` for every text.
- When running locally, you can customize the code with your logic in the `_run` function.

In [9]:
class SentimentAnalysisTool(BaseTool):
    name: str ="Sentiment Analysis Tool"
    description: str = ("Analyzes the sentiment of text "
         "to ensure positive and engaging communication.")
    
    def _run(self, text: str) -> str:
        # Your custom code tool goes here
        return "positive"

In [10]:
sentiment_analysis_tool = SentimentAnalysisTool()

## Creating Tasks

- The Lead Profiling Task is using crewAI Tools.

In [11]:
lead_profiling_task = Task(
    description=(
        "Conduct an in-depth analysis of {lead_name}, "
        "a company in the {industry} sector "
        "that recently showed interest in our solutions. "
        "Utilize all available data sources "
        "to compile a detailed profile, "
        "focusing on key decision-makers, recent business "
        "developments, and potential needs "
        "that align with our offerings. "
        "This task is crucial for tailoring "
        "our engagement strategy effectively.\n"
        "Don't make assumptions and "
        "only use information you absolutely sure about."
    ),
    expected_output=(
        "A comprehensive report on {lead_name}, "
        "including company background, "
        "key personnel, recent milestones, and identified needs. "
        "Highlight potential areas where "
        "our solutions can provide value, "
        "and suggest personalized engagement strategies."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)

- The Personalized Outreach Task is using your custom Tool `SentimentAnalysisTool`, as well as crewAI's `SerperDevTool` (search_tool).

In [12]:
personalized_outreach_task = Task(
    description=(
        "Using the insights gathered from "
        "the lead profiling report on {lead_name}, "
        "craft a personalized outreach campaign "
        "aimed at {key_decision_maker}, "
        "the {position} of {lead_name}. "
        "The campaign should address their recent {milestone} "
        "and how our solutions can support their goals. "
        "Your communication must resonate "
        "with {lead_name}'s company culture and values, "
        "demonstrating a deep understanding of "
        "their business and needs.\n"
        "Don't make assumptions and only "
        "use information you absolutely sure about."
    ),
    expected_output=(
        "A series of personalized email drafts "
        "tailored to {lead_name}, "
        "specifically targeting {key_decision_maker}."
        "Each draft should include "
        "a compelling narrative that connects our solutions "
        "with their recent achievements and future goals. "
        "Ensure the tone is engaging, professional, "
        "and aligned with {lead_name}'s corporate identity."
    ),
    tools=[sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

## Creating the Crew

In [13]:
crew = Crew(
    agents=[sales_rep_agent, 
            lead_sales_rep_agent],
    
    tasks=[lead_profiling_task, 
           personalized_outreach_task],
	
    verbose=2,
	memory=True
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [14]:
inputs = {
    "lead_name": "DeepLearningAI",
    "industry": "Online Learning Platform",
    "key_decision_maker": "Andrew Ng",
    "position": "CEO",
    "milestone": "product launch"
}

result = crew.kickoff(inputs=inputs)

 [DEBUG]: == Working Agent: Sales Representative
 [INFO]: == Starting Task: Conduct an in-depth analysis of DeepLearningAI, a company in the Online Learning Platform sector that recently showed interest in our solutions. Utilize all available data sources to compile a detailed profile, focusing on key decision-makers, recent business developments, and potential needs that align with our offerings. This task is crucial for tailoring our engagement strategy effectively.
Don't make assumptions and only use information you absolutely sure about.


> Entering new CrewAgentExecutor chain...
I need to gather detailed information about DeepLearningAI to create a comprehensive report that highlights potential opportunities for our company.

Action: Read a file's content
Action Input: {"file_path": "DeepLearningAI_company_background.txt"} 

I encountered an error while trying to use the tool. This was the error: [Errno 2] No such file or directory: 'DeepLearningAI_company_background.txt'.
 Tool 

Thought: I now know the final answer

Final Answer: 

Comprehensive Report on DeepLearningAI:

Company Background:
DeepLearningAI is an education technology company founded by Andrew Ng, a global leader in AI. The company is dedicated to making world-class AI education accessible to people around the globe. DeepLearningAI's main headquarters is located at 2445 Faber Pl, Palo Alto, California 94303, US. The company has employees across 6 continents, including Asia. They offer AI courses and specializations to over 7 million people, building a foundation of machine learning and AI skills.

Key Personnel:
- Andrew Ng: Founder of DeepLearningAI, Managing General Partner of AI Fund, and Executive Chairman of Landing AI.
- Ted Greenwald: Engineering at DeepLearningAI.
- Joe Chen: Engineering at DeepLearningAI.
- Bradley Reynolds: Chief Strategy Officer (CSO) / SVP of AI at DeepLearningAI.
- Kirsty Tan: Dream. Believe.

Recent Milestones and Business Developments:
1. The Batch AI News and Ins

- Display the final result as Markdown.

In [15]:
from IPython.display import Markdown
Markdown(result)

Subject: Elevating AI Education with DeepLearningAI's Recent Achievements

Dear Andrew Ng,

I hope this message finds you well. I wanted to take a moment to commend DeepLearningAI on the recent milestones achieved in advancing AI education globally. The news of closing $190M for the new fund is truly remarkable and a testament to the dedication and innovation driving DeepLearningAI's mission.

As a global leader in AI education, DeepLearningAI's commitment to making world-class AI education accessible to millions is truly inspiring. The launch of AI avatars and generative AI platforms further showcases your dedication to pushing boundaries and enhancing the learning experience for individuals worldwide.

At CrewAI, we share the same passion for empowering individuals with AI skills and enhancing the education space with cutting-edge technologies. Our solutions are designed to complement DeepLearningAI's initiatives by providing advanced tools and personalized engagement strategies to cater to the diverse learning needs of your users.

We believe that by collaborating with DeepLearningAI, we can collectively drive positive change in the AI education landscape and continue to make a meaningful impact on the global AI community. I would be thrilled to discuss how our solutions can support your goals and contribute to the success of DeepLearningAI's mission.

Looking forward to the opportunity to connect and explore potential synergies between CrewAI and DeepLearningAI. Thank you for your time and consideration.

Warm regards,

[Your Name]
Lead Sales Representative
CrewAI

---

Subject: Enabling AI Empowerment with CrewAI's Solutions

Dear Andrew Ng,

I trust this message finds you in good spirits. I am reaching out to express my admiration for DeepLearningAI's unwavering commitment to advancing AI education globally. The recent achievements, including the launch of AI avatars and generative AI platforms, are a testament to your dedication and innovation in the education technology space.

As a company deeply invested in empowering individuals with AI skills, CrewAI recognizes the importance of providing cutting-edge solutions to support educational initiatives like those undertaken by DeepLearningAI. Our advanced tools and personalized engagement strategies are designed to enhance learning experiences and cater to the evolving needs of learners worldwide.

Collaborating with DeepLearningAI presents an exciting opportunity to combine our expertise and resources to further elevate the AI education landscape. Together, we can create impactful solutions that drive innovation, foster creativity, and empower individuals to succeed in the rapidly evolving field of AI.

I would be delighted to explore how CrewAI's solutions can complement DeepLearningAI's vision and contribute to the continued success of your educational initiatives. Thank you for considering this partnership opportunity, and I look forward to the possibility of working together.

Warm regards,

[Your Name]
Lead Sales Representative
CrewAI